<a href="https://colab.research.google.com/github/ChanderValasai/gender-pay-gap-audit/blob/main/notebooks/03_hypothesis_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/data-analysis-projects/pakistan-tech-pay-gap'
os.chdir(PROJECT_PATH)

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('data/processed/cleaned_survey_2025.csv')
df['LogComp'] = np.log(df['ConvertedCompYearly'])

pk = df[df['CountryGroup'] == 'Pakistan']['ConvertedCompYearly']
row = df[df['CountryGroup'] == 'Rest of World']['ConvertedCompYearly']
pk_log = df[df['CountryGroup'] == 'Pakistan']['LogComp']
row_log = df[df['CountryGroup'] == 'Rest of World']['LogComp']

Mounted at /content/drive


## Significance Threshold

α = 0.05 for all hypothesis tests in this phase. This is the conventional threshold for exploratory
business/social-science analysis and is set here before viewing test results, to avoid adjusting the
threshold post-hoc to fit findings.

Given the large sample size for Rest of World (n=23,000), statistical significance alone is expected
to be achievable even for small effects. Effect size (Cohen's d) will be reported alongside p-values
to assess practical significance, not just statistical detectability.

In [2]:
# 1. Welch's t-test on raw values
t_raw, p_raw = stats.ttest_ind(pk, row, equal_var=False)
print(f"Welch's t-test (raw): t={t_raw:.3f}, p={p_raw:.6f}")

# 2. Welch's t-test on log-transformed values
t_log, p_log = stats.ttest_ind(pk_log, row_log, equal_var=False)
print(f"Welch's t-test (log): t={t_log:.3f}, p={p_log:.6f}")

# 3. Mann-Whitney U (non-parametric, no distribution assumption)
u_stat, p_mw = stats.mannwhitneyu(pk, row, alternative='two-sided')
print(f"Mann-Whitney U: U={u_stat:.1f}, p={p_mw:.6f}")

Welch's t-test (raw): t=-37.323, p=0.000000
Welch's t-test (log): t=-19.094, p=0.000000
Mann-Whitney U: U=256110.0, p=0.000000


In [3]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

d_raw = cohens_d(row, pk)   # Rest of World vs Pakistan, raw
d_log = cohens_d(row_log, pk_log)  # same, log-transformed

print(f"Cohen's d (raw): {d_raw:.3f}")
print(f"Cohen's d (log): {d_log:.3f}")

Cohen's d (raw): 0.852
Cohen's d (log): 1.867


## Hypothesis Testing Summary (Unadjusted)

**Result:** All three tests (Welch's t-test raw, Welch's t-test log, Mann-Whitney U) reject H0
at p < .001. The unadjusted salary gap between Pakistan and Rest-of-World respondents is
statistically significant.

**Effect size:** Cohen's d = 0.852 (raw scale), 1.867 (log scale). The log-scale estimate is
considered more reliable here, since raw-scale variance is inflated by extreme high earners in
the Rest-of-World group, artificially compressing the raw d. By conventional benchmarks
(0.2/0.5/0.8 = small/medium/large), this represents a large effect on both scales.

**Interpretation caution:**
- Statistical significance here is unsurprising given n=23,000 on one side — even trivial
  differences would likely reach significance at this sample size. The large effect size is
  what makes this finding substantively meaningful, not the p-value alone.
- This is still the UNADJUSTED gap. It has not controlled for experience (YearsCode), which
  could plausibly explain some or all of this difference if Pakistan's respondents skew junior
  relative to Rest-of-World.
- This measures nominal USD compensation, not purchasing power — a large nominal gap is
  expected given cost-of-living differences and does not by itself indicate discrimination.
- Correlation/association ≠ causation: this establishes an association between country group
  and compensation, not a causal claim about *why* the gap exists.

**Next step:** Phase 7 will test whether this gap persists after controlling for YearsCode
(and potentially DevType/role), which is the central question this project was designed to answer.